# 2장. RAG 시스템 구축 — Retrieve (검색기 생성)

**Retrieve**는 RAG 파이프라인의 다섯 번째 단계입니다.  
저장된 Chroma DB를 불러와 `as_retriever()`로 검색기를 생성합니다.

**기존 DB 불러오기** : `documents=` 없이 경로만 지정하면 기존 데이터를 로드합니다.

**검색 타입 비교:**

| 타입 | 설명 | 파라미터 |
|:---|:---|:---|
| `similarity` (기본) | 유사도 높은 순 상위 k개 반환 | `k` |
| `similarity_score_threshold` | 유사도 점수 임계값 이상만 반환 | `score_threshold` |
| `mmr` | 관련성 + 다양성 함께 고려 | `k` |

> `hnsw:space: cosine` — ChromaDB 내부 검색 알고리즘이 **코사인 유사도**를 기준으로 거리를 계산하도록 설정합니다.  
> (옵션: `"l2"` 유클리드 거리, `"ip"` 내적)

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace=underlying_embeddings.model
)


/Users/dohyunkim/Documents/langchain-for-ai-agent/.venv/lib/python3.13/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [2]:
from langchain_chroma import Chroma

CHROMA_PATH = "./chroma_db"


db = Chroma(
    collection_name="rag_collection",
    embedding_function=cached_embedder,
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"}  # 옵션: "cosine", "l2" (유클리드), "ip" (내적)
)


In [4]:
# 특징: documents= 인자가 없습니다. 
# 즉, 새로운 문서를 추가하는 게 아니라 기존에 저장된 데이터를 불러와서 검색하기 위한 용도로 사용합니다.
db = Chroma(
    collection_name="rag_collection",
    embedding_function=cached_embedder,
    # embedding_function: 검색 시 사용자가 던지는 질문(Query)을 숫자로 바꿀 때 사용할 모델입니다.
    # cached_embedder를 사용하면 질문이 이전과 같을 경우 캐시된 값을 재사용합니다.

    persist_directory=CHROMA_PATH,
    # persist_directory: 벡터 데이터가 실제로 저장되어 있는 물리적 경로입니다.
    # 이 경로에 이미 데이터가 있다면 해당 데이터를 로드하고, 없다면 빈 데이터베이스를 초기화합니다.
    # (주의: documents 인자가 없으므로 이 코드 실행만으로는 새로운 문서가 저장되지 않습니다.)

    collection_metadata={"hnsw:space": "cosine"}  # 옵션: "cosine", "l2" (유클리드), "ip" (내적)

    # hnsw: ChromaDB가 내부적으로 사용하는 고속 검색 알고리즘(Hierarchical Navigable Small World)의 약자입니다.
    # space: 검색 공간에서 벡터 간의 '거리'를 측정하는 방식입니다.
    # "cosine"은 벡터 간의 각도를 측정하여 의미적 유사성을 계산하는 가장 일반적인 방식입니다.
)

In [5]:

query = "Tesla 투자 비중이 얼마나 되나요?"
results = db.similarity_search(query)
# 벡터스토어에 있는 문서와 유사도를 계산하고, 유사도가 높은 순서대로 검색 결과를 반환한다

print(len(results))

print(f"검색된 문서 내용:\n{results[0].page_content}")
# 가장 유사한 문서의 내용을 반환한다
# 이 내용을 LLM에게 전달해서 답변을 생성할 수 있다
# 이것이 RAG 

4
검색된 문서 내용:
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%


In [7]:
# 검색기 생성
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.4}
    # score_threshold: 유사도 점수가 0.4 이상인 문서만 검색하겠다는 의미
    # 따라서 유사도 점수를 높이면 검색 결과가 없을 수도 있다.
)

results = retriever.invoke(query)

print(len(results))

print(results)


No relevant docs were retrieved using the relevance score threshold 0.4


0
[]


In [8]:
# 검색기 생성
retriever2 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3,}

)


results2 = retriever2.invoke(query)

print(len(results2))

print(results2[0].page_content)

3
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%
